# పాఠం 04 - టూల్ ఉపయోగం డిజైన్ నమూనా

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **టూల్ ఉపయోగం** డిజైన్ నమూనాను నేర్చుకుంటారు. ఇందులో కవర్ చేస్తాము:

- `@tool` డెకొరేటర్ మరియు టైప్డ్ పరిమాణాలతో ఫంక్షన్ టూల్స్ నిర్వచించడం
- మోడల్‌కు ప్రతి టూల్ ఏం చేస్తుందో తెలియజేసే టూల్ స్కీమాలు అందించడం
- `approval_mode` ద్వారా టూల్ ఎగ్జిక్యూషన్ నియంత్రించడం
- Pydantic మోడల్స్ మరియు `response_format` ద్వారా **సంరచనాత్మక అవుట్పుట్** తిరిగి ఇవ్వడం

సన్నివేశం ఒక **ప్రయాణ బుకింగ్ ఏజెంట్** సందర్భం, ఇది గమ్యస్థానాలను చూసుకోవడం, అందుబాటులో ఉన్నదాన్ని తనిఖీ చేయడం, మరియు విమాన సమాచారాన్ని పొందగలదు.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool డెకొరేటర్‌తో టూల్స్ నిర్వచనము

`@tool` డెకొరేటర్ ఒక సాధారణ పైథాన్ ఫంక్షన్‌ను ఏజెంట్ కాల్ చేయగల టూల్ గా మార్చుతుంది. ముఖ్యమైన విషయాలు:

- **డాక్ స్ట్రింగ్** మోడల్ చూస్తున్న టూల్ వివరణ అవుతుంది.
- **టైప్ అనోటేషన్లు** (వివరణలతో ఉన్న `Annotated` సహా) టూల్ స్కీమాను నిర్వచిస్తాయి.
- `approval_mode` ప్రతి కాల్ నడిచే ముందు యూజర్ ఆమోదించాల్సిన అవసరాన్ని నియంత్రిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## బహువిధ సాధనాలతో ఏజెంట్ తయారు చేయడం

మూడు సాధనలను client కు అందించండి, తద్వారా మోడల్ అవసరమైన దాన్ని ఉపయోగించి వినియోగదారు ప్రశ్నకు సమాధానం ఇవ్వగలదు.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## సాధనాలతో నిర్మిత అవుట్పుట్

`response_format` ని Pydantic మోడల్‌గా సెట్ చేయడం ద్వారా, ఏజెంట్ స్వేచ్ఛా వచనం కాకుండా బాగా టైప్ చేసిన JSON వస్తువును తిరిగి ఇవ్వడానికి బలవంతం చేయబడుతుంది. డౌన్‌స్ట్రీమ్ కోడ్ ఫలితాన్ని ప్రోగ్రామానుగతంగా వినియోగించుకోవాలనుకుంటే ఇది ఉపయోగకరం.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## టూల్ ఆమోద నమూనాలు

`@tool` పై ఉన్న `approval_mode` పారామీటర్ టూల్ కాల్స్ నడపముందు మానవ ఆమోదం అవసరమో కాదో నియంత్రిస్తుంది:

| మోడ్ | ప్రవర్తన |
|---|---|
| `"never_require"` | టూల్ ఆటోమేటిక్‌గా అమలు అవుతుంది — యూజర్ ధృవీకరణ అవసరం లేదు. |
| `"always_require"` | ప్రతి కాల్ అమలు కాబోయే ముందు యూజర్ ఆమోదం పొందాలి. |

పక్క వచ్చిన ప్రభావాలు ఉన్న పరికరాలకు (ఉదా: ఫ్లైట్ బుకింగ్, క్రెడిట్ కార్డ్ చార్జ్ చేయడం) `"always_require"` ఉపయోగించండి, తద్వారా ఒక మానవుడు నిలవగలుగుతాడు.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## సారాంశం

ఈ పాఠంలో మీరు తెలుసుకున్నారు ఎలా:

1. టైప్ చేసిన పారామీటర్లు మరియు టూల్ స్కీమాగా పనిచేసే డాక్స్ట్రింగ్లతో `@tool` డెకరేటర్ ఉపయోగించి **టూల్స్ నిర్వచించాలి**.
2. ఏజెంట్ సంక్లిష్ట ప్రశ్నలకు సమాధానం చెప్పటానికి వారుసవద్దగా పిలవగల విధంగా **బహుళ టూల్స్‌ను కూర్చేయాలి**.
3. `response_format` గా Pydantic మోడల్‌ను పంపించి **సంఘటిత ఔట్‌పుట్‌ను తిరిగి ఇవ్వాలి**.
4. స్పర్శకమైన ఆపరేషన్ల కోసం మానవ జోల్పు ఉంచేందుకు `approval_mode` తో **టూల్ మంజూరు నియంత్రణ** చేయాలి.

ఈ నమూనాలు నమ్మకమైన, ఉత్పత్తి-సిద్ధమైన ఏజెంట్లను సురక్షితంగా బయటి వ్యవస్థలతో ఇంచుమించు పనిచేసే విధంగా తయారుచేయడానికి మూలస్తంభంగా ఉంటాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
